# 12.6 線性搜尋與 index() 例外防範

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_12-6_linear_search_and_index_defense.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 Chapter 8 串列索引操作、`index()` 方法與 Chapter 6 迴圈中斷機制。

---

### 學習導覽：地毯式尋寶與安全防護——搜尋演算法的原點

在電腦科學與資訊競賽中，除了將資料由小到大排好之外，另一項高達 90% 以上時間都在進行的超級任務就是——**搜尋（Searching）**！
「使用者輸入了一組身分證字號，請找出他在不在會員清單中？在第幾筆紀錄？」
「在海量的測量數據中，有沒有出現異常數值 999？它出現在哪些時間點？」

面對一群尚未排序、或者無法預知規律的資料，我們最原始、最直觀、也最不可或缺的搜尋手段，就是**線性搜尋（Linear Search）**：
就像拿著手電筒，在黑暗的走廊上一間一間敲門，直到找到目標為止。

然而，Python 內建的 `list.index()` 方法雖然方便，卻隱藏著一顆致命的定時炸彈：
**「只要你要找的目標不在串列中，它不會回傳 -1，而是當場引發 `ValueError` 讓整個程式直接崩潰！」**
在 APCS 線上評判系統（OJ）中，只要遇到一筆找不到的測試資料，你的程式就會直接噴出 Runtime Error (RE)，痛失全部得分！

在本單元中，我們將透過 6 個平緩的微階梯，建立堅固的搜尋思維與防禦架構：
1. **12.6.1 線性搜尋（Linear Search）原理**：無序資料的地毯式逐一走訪。
2. **12.6.2 時間複雜度 $O(N)$ 分析**：最佳情況、最差情況與平均比對次數。
3. **12.6.3 `list.index(x)` 的致命陷阱**：找不到數值時引發 `ValueError` 崩潰。
4. **12.6.4 成員檢查防禦安全查詢**：以 `if x in a:` 搭配 `a.index(x)` 確保安全查詢。
5. **12.6.5 手刻線性搜尋**：回傳第一個匹配索引，找不到時回傳 -1 的經典慣用法。
6. **12.6.6 多重目標搜尋**：收集所有匹配條件的索引清單與帶條件走訪。

讓我們築起最穩健的安全防護網，踏入搜尋演算法的世界！

### 12.6.1 線性搜尋（Linear Search）原理：無序資料的地毯式逐一走訪

#### 1. 生活故事比喻：黑暗更衣室裡尋找自己的球鞋
想像體育課下課後，學校體育館的置物間突然停電，裡面一片漆黑。地上橫七豎八擺著五十雙各班同學踢散的球鞋，沒有任何編號或排序規律。
你手裡拿著一支微弱的手電筒，想要找回自己那雙藍色慢跑鞋。
在完全不知道鞋子放在哪裡的情況下，你能怎麼做？
你只能從門口的第一雙鞋開始看：「這雙是我的嗎？不是。下一雙是我的嗎？不是……」一雙接著一雙，地毯式地向房間深處逐一查驗。
如果在第十雙找到了，你興奮地拿著鞋子離開；如果查驗完整整五十雙都沒看到，你才能確定鞋子不在置物間裡。
這種**「從頭到尾、一個不漏、循序逐一比對」**的尋找過程，就是**線性搜尋（Linear Search）**，也俗稱**循序搜尋（Sequential Search）**。

#### 2. 底層運作機制：單向走訪與提早收工
線性搜尋的內部邏輯極度純粹：
1. 將索引指標 $i$ 從 $0$ 開始出發。
2. 比對當前元素：`if a[i] == target:`
   - 若相等：目標命中！立刻紀錄當前位置 $i$，並使用 `break` 提早結束走訪。
   - 若不相等：將指標向前推進 $i = i + 1$，繼續檢查下一個隔間。
3. 若指標已經走過串列末端（$i \ge N$）仍然沒有命中，宣告目標不存在。

#### 3. 初學者常見陷阱：沒找完全部就急著宣判「找不到」
初學者在手刻搜尋時，常常寫出嚴重的邏輯失誤：
```python
# 致命錯誤！
for x in nums:
    if x == target:
        print("找到了！")
    else:
        print("找不到！")  # 崩潰！剛比對第 0 個不是，就立刻大喊找不到！
```
請記住：**「只要有一個是，就是找到了；但要宣判找不到，必須看完最後一個人才能下定論！」** 絕不能在 `else` 裡面立刻宣告失敗。

#### 4. APCS 實戰視野
雖然線性搜尋看似笨拙，但它是面對「無序資料（Unordered Data）」唯一正確且保證能找到答案的演算法。在資料量較小（例如 $N \le 1,000$）或資料隨時動態插入未排序時，線性搜尋是最容易編寫且萬無一失的解題利器。

In [ ]:
# 範例 12.6.1：線性搜尋基礎原理演示
# 一組無序的學生成績清單
scores = [65, 82, 90, 74, 99, 88, 55]
target = 99  # 我們想要尋找的滿分目標
print("待搜尋成績串列：", scores)
print("目標數值：", target)

# 啟動手電筒地毯式線性搜尋
found_index = None

for i in range(len(scores)):
    print(f"  檢查索引 {i}: 數值為 {scores[i]} ...", end=" ")
    if scores[i] == target:
        print("🎯 命中目標！提早收工！")
        found_index = i
        break  # 找到立刻跳出迴圈，不再浪費時間檢查後方
    else:
        print("不是目標，繼續往後看")

if found_index is not None:
    print(f"\n搜尋成功！目標 {target} 位於索引 {found_index}。")
else:
    print(f"\n搜尋失敗，目標 {target} 不存在於串列中。")

In [ ]:
# 填空題 12.6.1：循序尋找特定水果
# 任務：在水果攤清單中循序尋找 "cherry"，記錄其索引位置。
fruits = ["apple", "banana", "cherry", "date", "fig"]
target_fruit = "cherry"

match_idx = None
for i in range(len(fruits)):
    if fruits[i] == ___:
        match_idx = i
        # 命中後請立刻中斷迴圈
        ___

print(f"'{target_fruit}' 的位置在索引：", match_idx)

In [ ]:
# ==========================================
# [4] Code 練習題 12.6.1
# 任務說明：
# 給定一組無序整數清單 raw_data 與目標值 target。
# 請使用 for 迴圈逐一比對進行線性搜尋，
# 1. 印出總共比對了幾次才找到目標（或找完全部）。
# 2. 印出目標所在的索引位置（若不存在印出 "找不到"）。
#
# 【公開測試資料 1】
# raw_data = [42, 18, 95, 33, 70]
# target = 95
# 預期輸出：
# 比對次數： 3
# 目標索引： 2
#
# 【公開測試資料 2】
# raw_data = [10, 20, 30]
# target = 999
# 預期輸出：
# 比對次數： 3
# 目標索引： 找不到
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
raw_data = [42, 18, 95, 33, 70]
target = 95

comp_count = 0
found_pos = "找不到"

for i in range(len(raw_data)):
    comp_count += 1
    if raw_data[i] == target:
        found_pos = i
        break

print("比對次數：", comp_count)
print("目標索引：", found_pos)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.6.1
# 任務說明：
# 某量測儀器連續記錄了 8 筆水溫：temps = [24, 25, 24, 28, 31, 29, 32, 30]
# 請使用線性搜尋，找出「水溫第一次超過 30 度（> 30）」的時間點（索引位置），
# 並印出該高溫數值與其索引。
# 若全程未超過 30 度，則印出 "水溫安全無超標"。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
temps = [24, 25, 24, 28, 31, 29, 32, 30]
first_hot_idx = None

for i in range(len(temps)):
    if temps[i] > 30:
        first_hot_idx = i
        break

if first_hot_idx is not None:
    print(f"首次超標溫度: {temps[first_hot_idx]} 度，出現在索引 {first_hot_idx}")
else:
    print("水溫安全無超標")

### 12.6.2 時間複雜度 $O(N)$ 分析：最佳情況、最差情況與平均比對次數

#### 1. 生活故事比喻：鑰匙串開門的運氣好壞
想像你下班回家，手中有一串剛好串著 10 支一模一樣外觀的鑰匙。
你不知道哪一把才是大門鑰匙，只能插進鎖孔一把一把旋轉試探：
- **超級大好運（最佳情況，Best Case）**：隨手拿起的第一把鑰匙插進去，喀嚓一聲門開了！只試了 1 次，運氣好到爆表！
- **倒楣透頂（最差情況，Worst Case）**：前 9 把鑰匙通通轉不動，直到試到最後第 10 把鑰匙才打開；或者更慘——這串鑰匙根本不是你家的，你白白試滿了 10 次才發現！
- **一般平均（平均情況，Average Case）**：長遠來看，平均大約要試 5 次左右（$N / 2$ 次）。
在電腦科學中，我們評估一個演算法的好壞，不能單靠運氣，必須精確推導它的時間複雜度！

#### 2. 底層運作機制：大 $O$ 符號下的 $O(N)$ 定義
- **最佳情況（Best Case）**：$O(1)$。目標剛好躺在第 0 個位置，比對 1 次立即命中。
- **最差情況（Worst Case）**：$O(N)$。目標位於最後一個位置 $N-1$，或者目標根本不存在，演算法必須把全體 $N$ 個元素全盤看過一遍。
- **平均情況（Average Case）**：大約比較 $\frac{N+1}{2}$ 次。在漸進分析中，常數係數 $\frac{1}{2}$ 被忽略，依然屬於 $O(N)$ 等級！
資訊科學中討論複雜度時，**預設永遠以「最差情況（Worst Case）」為基準**！因此線性搜尋的時間複雜度被統一定義為 **$O(N)$**。

#### 3. 初學者常見陷阱：以為 $O(N)$ 在任何時候都夠快
當資料量 $N = 100$ 時，$O(N)$ 比對 100 次，在電腦上看來是幾微秒的微光一瞬。
但如果你的資料庫有 $N = 10,000,000$（一千萬筆會員紀錄），且每秒鐘有 1,000 個人在網站搜尋，每次搜尋都要線性走訪一千萬次，伺服器 CPU 會瞬間飆到 100% 當機！這也是為什麼後續第 12.7 節我們必須學習二分搜尋法的原因。

#### 4. APCS 實戰視野
APCS 筆試觀念題常考「線性搜尋平均比對次數（$(N+1)/2$）」與「最差比對次數（$N$）」。深刻理解這三個情況的數學定義，是拿到演算法觀念題滿分的保證。

In [ ]:
# 範例 12.6.2：線性搜尋三種情況比對次數實測
arr = [10, 20, 30, 40, 50, 60, 70, 80]
N = len(arr)
print(f"測試串列長度 N = {N}，內容: {arr}")

def count_comparisons(data, target):
    comparisons = 0
    for x in data:
        comparisons += 1
        if x == target:
            return comparisons
    return comparisons

# 1. 最佳情況（Best Case）：目標在首位
best = count_comparisons(arr, 10)
print(f"1. 最佳情況（找 10）：比對次數 = {best} 次 (O(1))")

# 2. 最差情況（Worst Case）：目標在末位或不存在
worst_end = count_comparisons(arr, 80)
worst_none = count_comparisons(arr, 999)
print(f"2. 最差情況（找 80）：比對次數 = {worst_end} 次 (O(N))")
print(f"   最差情況（找不存在值）：比對次數 = {worst_none} 次 (O(N))")

# 3. 平均比對次數理論值驗證：所有元素都找一次的平均
total_steps = sum(count_comparisons(arr, x) for x in arr)
avg_steps = total_steps / N
print(f"3. 實測平均比對次數 = {avg_steps:.1f} 次，理論公式 (N+1)/2 = {(N+1)/2:.1f} 次")

In [ ]:
# 填空題 12.6.2：計算最差情況比對次數
# 任務：若清單中有 120 筆資料，計算線性搜尋在「最差情況」下的比對次數。
data_size = 120

# 最差情況下，比對次數等於資料總長度 N
worst_case_steps = ___

print(f"資料量為 {data_size} 時，線性搜尋最差需比對 {worst_case_steps} 次。")

In [ ]:
# ==========================================
# [4] Code 練習題 12.6.2
# 任務說明：
# 給定一組串列 test_arr 與 3 個搜尋目標 targets。
# 請寫出程式，依序統計尋找這 3 個目標各自花了幾次比對，
# 並輸出各目標的比對次數清單。
#
# 【公開測試資料 1】
# test_arr = [5, 15, 25, 35, 45]
# targets = [5, 35, 100]
# 預期輸出：
# 各目標比對次數： [1, 4, 5]
#
# 【公開測試資料 2】
# test_arr = [1, 2, 3]
# targets = [2, 99]
# 預期輸出：
# 各目標比對次數： [2, 3]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
test_arr = [5, 15, 25, 35, 45]
targets = [5, 35, 100]

def get_steps(arr, t):
    steps = 0
    for x in arr:
        steps += 1
        if x == t:
            break
    return steps

results = [get_steps(test_arr, t) for t in targets]
print("各目標比對次數：", results)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.6.2
# 任務說明：
# 某系統維護了一串 6 個使用者的 ID：ids = [101, 102, 103, 104, 105, 106]
# 假設每個使用者被查詢的機率完全均等。
# 請寫一段程式碼，模擬查詢清單中的每一個 ID，
# 計算出整體的「平均比對次數」，並印出格式化結果（保留小數一位）。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
ids = [101, 102, 103, 104, 105, 106]
total_steps = 0
for target in ids:
    for idx, val in enumerate(ids):
        if val == target:
            total_steps += (idx + 1)
            break

avg = total_steps / len(ids)
print(f"長度 {len(ids)} 串列的平均比對次數: {avg:.1f} 次")

### 12.6.3 `list.index(x)` 方法的致命陷阱：找不到數值時引發 `ValueError` 崩潰

#### 1. 生活故事比喻：暴躁的圖書館櫃台管理員
想像你去圖書館櫃台查詢一本絕版書：「請問館內有《Python 演算法天書》嗎？有的話在第幾號書架？」
- 如果館內有這本書，管理員親切地告訴你：「在 3 號書架！」（回傳索引 `3`）。
- 但如果館內「根本沒有這本書」，理想的管理員應該禮貌地說：「抱歉，本館查無此書，回傳 -1 代表找不到」。
但 Python 內建的 `list.index()` 就像一位脾氣極其暴躁的管理員！一旦電腦在串列裡翻遍了找不到你指定的東西，他不會客氣地說沒有，而是當場「掀翻桌子大吼大叫，觸發火警警報（拋出 `ValueError`）」，讓整棟圖書館的人員全數緊急疏散（整個程式直接被作業系統強制中斷關閉）！

#### 2. 底層運作機制：`ValueError: x is not in list` 的報錯本質
在 Python 官方設計中，`list.index(x)` 的合約極為嚴苛：
- 若 `x` 存在：回傳 `x` 在串列中第一次出現的整數索引（`0 <= index < len(list)`）。
- 若 `x` 不存在：**直接拋出 `ValueError: <value> is not in list` 異常！**
這與其他程式語言（如 C++ 的 `find` 回傳 end 迭代器，或 JavaScript 的 `indexOf` 回傳 `-1`）完全不同！
許多從其他語言轉戰 Python 的工程師或初學者，習慣性地寫出：
```python
pos = my_list.index(target)
if pos == -1: ...  # 根本走不到這一步！在上一行就已經當機噴錯了！
```

#### 3. 初學者常見陷阱：直接在未防護的情況下呼叫 `.index()`
初學同學最常吞下 Runtime Error 的寫法：
```python
scores = [80, 90, 70]
# 致命！若輸入 100 分，立刻引發崩潰！
idx = scores.index(100)  # ValueError: 100 is not in list
```
如果不加任何安全防護措施就直接呼叫 `.index()`，這段程式碼在正式競賽中就像一顆沒有安全插銷的手榴彈，只要裁判出了一筆邊界不存在的測資，全題當場 0 分！

#### 4. APCS 實戰視野
APCS 線上測驗（OJ）最喜歡暗藏「搜尋目標根本不在測資中」的測試案例。認清 `.index()` 的拋出異常機制，是學會「安全查詢防禦」不可逾越的第一道認知防線。

In [ ]:
# 範例 12.6.3：親身體驗 list.index() 的成功與當機陷阱
names = ["Alice", "Bob", "Charlie", "David"]
print("名冊串列：", names)

# 1. 成功案例：目標存在
idx_bob = names.index("Bob")
print("成功找到 'Bob'，索引為：", idx_bob)

# 2. 致命案例示範：目標不存在
target_missing = "Eve"
print(f"\n嘗試搜尋不存在的名單 '{target_missing}' ...")

try:
    # 這裡會拋出 ValueError！
    idx_eve = names.index(target_missing)
    print("索引為：", idx_eve)
except ValueError as e:
    print(f"💥 捕捉到系統崩潰警報！錯誤訊息為：{e}")
    print("提醒：.index() 找不到元素時絕對不會回傳 -1，而是直接報錯！")

In [ ]:
# 填空題 12.6.3：防範 .index() 的 ValueError 異常
# 任務：使用 try-except 結構包裹 .index()，確保程式遇到不存在元素時不當機。
gadgets = ["phone", "tablet", "laptop"]
query = "smartwatch"

try:
    # 呼叫 gadgets 的 index 方法
    pos = gadgets.___(query)
    print("找到裝置，位置：", pos)
except ___:
    print(f"安全防護觸發：'{query}' 不在清單中，已成功攔截當機！")

In [ ]:
# ==========================================
# [4] Code 練習題 12.6.3
# 任務說明：
# 給定一組代碼清單 codes 與待查詢代碼 query。
# 請使用 try...except ValueError 結構安全查詢索引：
# 1. 若找到，印出 "找到代碼於索引：X"。
# 2. 若找不到引發 ValueError，攔截並印出 "查無此代碼"。
#
# 【公開測試資料 1】
# codes = ["C101", "C102", "C103"]
# query = "C102"
# 預期輸出：
# 找到代碼於索引： 1
#
# 【公開測試資料 2】
# codes = ["C101", "C102"]
# query = "C999"
# 預期輸出：
# 查無此代碼
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
codes = ["C101", "C102", "C103"]
query = "C102"

try:
    p = codes.index(query)
    print("找到代碼於索引：", p)
except ValueError:
    print("查無此代碼")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.6.3
# 任務說明：
# 某登入系統記錄了一批白名單 IP：whitelist = ["192.168.1.1", "10.0.0.1", "172.16.0.1"]
# 請設計一個安全驗證函數 check_ip_index(ip_list, target_ip)，
# 內部使用 try-except ValueError：
# 若 target_ip 在清單中，回傳其整數索引；若不在清單中，安全回傳字串 "DENIED"。
# 印出測試結果。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def check_ip_index(ip_list, target_ip):
    try:
        return ip_list.index(target_ip)
    except ValueError:
        return "DENIED"

whitelist = ["192.168.1.1", "10.0.0.1", "172.16.0.1"]
print("查詢合法 IP: 10.0.0.1 ->", check_ip_index(whitelist, "10.0.0.1"))
print("查詢非法 IP: 8.8.8.8 ->", check_ip_index(whitelist, "8.8.8.8"))

### 12.6.4 成員檢查防禦安全查詢：以 `if x in a:` 搭配 `a.index(x)` 確保程式不崩潰

#### 1. 生活故事比喻：先敲門確認有人在，再開門進屋
既然直接呼叫 `.index()` 會有掀桌當機的巨大風險，那我們該如何優雅地保護自己？
想像你來到一間包廂門口，如果你二話不說直接用力推開門，萬一裡面沒人或者上鎖了，你可能會直接撞得鼻青臉腫。
一個有禮貌且聰明的人會怎麼做？
他會**「先伸出手敲敲門，問一聲：請問有人在嗎？（`if x in a:`）」**。
- 只有當裡面傳出回應「有人！（回傳 `True`）」時，他才放心地推門進去拿資料（呼叫 `a.index(x)`）。
- 如果敲門完全無人回應（回傳 `False`），他直接轉身離開，根本不會去硬推門！
這種**「先用 `in` 哨兵偵測，確認安全後再索引取值」**的雙重防線，是 Python 程式設計中最高雅、最常見的安全防禦模式！

#### 2. 底層運作機制：`in` 運算子的先導探路
Python 的 `in` 關鍵字在串列上執行時，本質上也是在串列內部進行一次快速走訪。
當我們寫出：
```python
if target in my_list:
    pos = my_list.index(target)
else:
    pos = -1
```
- 如果 `target in my_list` 為假，`else` 分支直接攔截，`.index()` 根本沒有被執行的機會，自然 100% 免疫 `ValueError`！
- 如果為真，保證 `target` 必定存在，此時呼叫 `.index(target)` 百分之百安全通過！

#### 3. 初學者常見陷阱：兩次走訪的效能代價
請注意：`in` 走訪了一次串列（$O(N)$），隨後的 `.index()` 又走訪了一次串列（$O(N)$）。
雖然在程式碼上多走了一次迴圈，但對於 APCS 數千筆以內的資料，這點微小常數開銷完全可以忽略，換來的是絕對不當機的強韌穩定性！

#### 4. APCS 實戰視野
在 APCS 實作考題中，`if item in container:` 搭配索引定位是最受推薦的標準查詢寫法。程式碼簡潔明瞭、邏輯一目了然，完全不需要撰寫冗長的例外處理架構，能有效降低考場焦慮感。

In [ ]:
# 範例 12.6.4：先 in 檢查、後 index 取值的安全查詢雙重防線
# 某商場的商品貨架清單
inventory = ["switch", "ps5", "xbox", "steam_deck"]
search_items = ["ps5", "gameboy", "switch"]

for item in search_items:
    print(f"正在搜尋商品: '{item}'")
    # 第一道防線：先用 in 運算子敲門確認
    if item in inventory:
        # 安全確認存在後，才放心呼叫 .index()
        shelf_pos = inventory.index(item)
        print(f"  ✅ 找到商品！位於第 {shelf_pos} 號貨架。")
    else:
        # 不存在時安全處理，絕不引發任何報錯崩潰！
        print(f"  ❌ 抱歉，商品 '{item}' 目前缺貨中。")

In [ ]:
# 填空題 12.6.4：安全查找學生成績座號
# 任務：查詢某位學生是否存在於名冊中，若在印出座號（索引），不在印出 -1。
students = ["Leo", "Max", "Nina", "Oscar"]
query_name = "Nina"

# 請使用 in 運算子先進行存在性檢查
if query_name ___ students:
    result_idx = students.___(query_name)
else:
    result_idx = -1

print(f"'{query_name}' 的查詢結果座號：", result_idx)

In [ ]:
# ==========================================
# [4] Code 練習題 12.6.4
# 任務說明：
# 給定一組會員編號清單 vip_members 與要查詢的 ID check_id。
# 請使用 "if ... in ..." 搭配 ".index()" 寫出安全查詢邏輯：
# 1. 若在清單中，印出 "會員存在，排位：X"（X 為索引）。
# 2. 若不在清單中，印出 "非會員，拒絕入場"。
#
# 【公開測試資料 1】
# vip_members = [1001, 1008, 1015, 1022]
# check_id = 1015
# 預期輸出：
# 會員存在，排位： 2
#
# 【公開測試資料 2】
# vip_members = [1001, 1008]
# check_id = 9999
# 預期輸出：
# 非會員，拒絕入場
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
vip_members = [1001, 1008, 1015, 1022]
check_id = 1015

if check_id in vip_members:
    print("會員存在，排位：", vip_members.index(check_id))
else:
    print("非會員，拒絕入場")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.6.4
# 任務說明：
# 某密室逃脫遊戲有一組密碼符號順序：symbols = ["star", "moon", "sun", "cloud", "rain"]
# 玩家輸入了一個符號猜測序列：guesses = ["sun", "fire", "star"]
# 請寫出程式碼，安全檢查 guesses 中的每一個符號：
# 若在 symbols 中，印出其順序位置；若不在，印出 "未知符號"。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
symbols = ["star", "moon", "sun", "cloud", "rain"]
guesses = ["sun", "fire", "star"]

for g in guesses:
    if g in symbols:
        print(f"符號 '{g}' 命中！順序為: {symbols.index(g)}")
    else:
        print(f"符號 '{g}' 為未知符號！")

### 12.6.5 手刻線性搜尋：回傳第一個匹配索引，找不到時回傳 -1 的經典慣用法

#### 1. 生活故事比喻：全功能客製化搜尋機
雖然 Python 內建了 `.index()` 和 `in`，但很多時候題目會有額外的特殊客製化要求：
- 「請找出第一個『大於 100』的數字在哪裡？」
- 「請找出第一個『以字母 A 開頭』的單字在哪裡？」
內建的 `.index(x)` 只能尋找「完全相等」的特定數值，面對這種「帶條件」的複雜搜尋，它就完全無能為力了。
這時，我們需要自己親手打造一部**手刻線性搜尋函數（Custom Linear Search）**！
在全體軟體工程與 APCS 競賽中，有一個通用的世界級默契：**「找到目標時回傳其 0-indexed 索引值；若走完全部皆找不到，一律回傳 `-1`」**！

#### 2. 底層運作機制：標準通用搜尋模板
讓我們封裝一個最標準、最經典的線性搜尋函數：
```python
def linear_search(arr, target):
    # 使用 enumerate 同時取得索引與數值
    for i, val in enumerate(arr):
        if val == target:
            return i  # 提早回傳命中索引
    return -1         # 迴圈正常結束代表查無此人，回傳 -1
```
這個模板的美妙之處在於：
1. **單一職責、零副作用**：它不依賴任何外部變數，輸入陣列與目標，輸出整數索引。
2. **絕無報錯風險**：找不到時優雅歸還 `-1`，呼叫端只要檢查 `if idx != -1:` 即可，徹底揮別 `ValueError` 的陰霾！

#### 3. 初學者常見陷阱：把 `return -1` 寫在迴圈內部
初學同學最容易犯的縮排大忌：
```python
def bad_search(arr, target):
    for i, val in enumerate(arr):
        if val == target:
            return i
        return -1  # 致命縮排錯誤！寫在 for 裡面，第 0 次不符合就直接回傳 -1 結束了！
```
請務必確認：**`return -1` 必須縮排在 `for` 迴圈的最外層！** 只有當整條迴圈完整跑完且沒有任何一次觸發命中時，才能執行這行代碼。

#### 4. APCS 實戰視野
手刻線性搜尋是 APCS 實作第二級的必考基礎模組。無論題目要找的是純數值、最大值第一次出現處、或是符合特定數學條件的門牌號碼，只要這個核心骨架在胸中，任何搜尋變形題都能信手拈來。

In [ ]:
# 範例 12.6.5：標準手刻線性搜尋函數（回傳索引或 -1）
def linear_search(arr, target):
    # 手刻標準線性搜尋
    # :param arr: 待搜尋清單
    # :param target: 搜尋目標
    # :return: 首次出現的索引位置，若無則回傳 -1
    for i, val in enumerate(arr):
        if val == target:
            return i
    return -1  # 迴圈走完仍未找到，回傳 -1

# 測試資料驗證
data = [15, 42, 88, 23, 70, 99, 31]
print("測試資料：", data)

# 案例 1：目標存在
res1 = linear_search(data, 88)
print(f"搜尋 88 的結果: {res1} (位於索引 {res1})")

# 案例 2：目標不存在
res2 = linear_search(data, 500)
print(f"搜尋 500 的結果: {res2} (安全回傳 -1，無任何異常！)")

# 示範慣用語法：利用 if res != -1 判定
if res1 != -1:
    print(f"確認命中！data[{res1}] == {data[res1]}")

In [ ]:
# 填空題 12.6.5：手刻帶條件搜尋（尋找第一個負數）
# 任務：手刻一個搜尋函數，找出串列中「第一個出現的負整數」的索引，若無負數回傳 -1。
def find_first_negative(numbers):
    for i, num in enumerate(numbers):
        # 條件：數值小於 0
        if num < ___:
            return ___
    # 走完全部都沒有負數
    return ___

test_list = [10, 25, -3, 8, -12, 40]
ans_idx = find_first_negative(test_list)
print("第一個負數出現在索引：", ans_idx)
print("數值為：", test_list[ans_idx])

In [ ]:
# ==========================================
# [4] Code 練習題 12.6.5
# 任務說明：
# 請撰寫一個函數 search_even(arr)，
# 循序搜尋串列 arr 中「第一個偶數（x % 2 == 0）」的索引位置。
# 若找到回傳該索引，若串列中完全沒有偶數則回傳 -1。
#
# 【公開測試資料 1】
# arr = [7, 13, 9, 14, 21, 8]
# 預期輸出：
# 第一個偶數索引： 3
#
# 【公開測試資料 2】
# arr = [1, 3, 5, 7]
# 預期輸出：
# 第一個偶數索引： -1
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def search_even(arr):
    for i, x in enumerate(arr):
        if x % 2 == 0:
            return i
    return -1

arr1 = [7, 13, 9, 14, 21, 8]
print("第一個偶數索引：", search_even(arr1))
arr2 = [1, 3, 5, 7]
print("第一個偶數索引：", search_even(arr2))

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.6.5
# 任務說明：
# 某通訊軟體有一批訊息字串清單：
# messages = ["Hello", "Good morning", "URGENT: server down!", "How are you?", "URGENT: backup failed"]
# 請撰寫函數 find_first_urgent(msg_list)，
# 找出第一則包含 "URGENT" 關鍵字的緊急訊息索引位置，若無緊急訊息回傳 -1。
# 印出該訊息的索引與完整內容。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def find_first_urgent(msg_list):
    for i, msg in enumerate(msg_list):
        if "URGENT" in msg:
            return i
    return -1

messages = ["Hello", "Good morning", "URGENT: server down!", "How are you?", "URGENT: backup failed"]
idx = find_first_urgent(messages)
if idx != -1:
    print(f"在索引 {idx} 發現第一則緊急警報：'{messages[idx]}'")
else:
    print("目前無任何緊急訊息")

### 12.6.6 多重目標搜尋：收集所有匹配條件的索引清單與帶條件走訪

#### 1. 生活故事比喻：全班段考 100 分的點名冊
在前面的單元中，我們的目標都是「找到第一個出現的人就收工」。
但如果今天期末考全班有 5 位同學都考了 100 分滿分，老師走進教室說：「請所有考 100 分的同學通通站起來，我要發獎勵卡片！」
這時，老師能看了第一個滿分的同學就立刻轉身回辦公室嗎？
**當然不能！**
老師必須拿著手電筒，把全班從 1 號到最後一號「完整走訪一遍」，把所有符合條件同學的座號一一記錄在一張清單小卡片上。
這種**「找出所有匹配項目、收集全部索引位置」**的操作，就是多重目標搜尋！

#### 2. 底層運作機制：列表生成式與條件收集
在 Python 中，多重目標搜尋有兩種優雅的實踐途徑：
- **途徑 A：傳統迴圈搭配 `.append()`**
  ```python
  matches = []
  for i, val in enumerate(data):
      if val == target:
        matches.append(i)
  ```
- **途徑 B：現代列表生成式（一行流）**
  ```python
  matches = [i for i, val in enumerate(data) if val == target]
  ```
這段代碼會走訪整個串列，每當條件成立，就把當前的索引 `i` 打包進全新的串列中。若完全沒有任何元素符合條件，回傳的就會是一個乾淨俐落的空串列 `[]`！

#### 3. 初學者常見陷阱：收集到了「數值」而不是「索引」
初學同學常常把 `val` 和 `i` 搞混：
```python
# 想要索引，卻寫成了：
matches = [x for x in data if x >= 60]  # 這收集到的是及格分數 [80, 95]，而不是座號！
```
請務必使用 `enumerate(data)` 同時拆解出 `i`（索引位置）與 `val`（數值內容），確認自己收集的是哪一個維度。

#### 4. APCS 實戰視野
APCS 題目經常要求：「輸出所有符合條件之測站編號，若無符合者輸出 -1」。
利用列表生成式收集索引清單：若 `len(matches) > 0` 則解包印出 `print(*matches)`；若 `len(matches) == 0` 則印出 `-1`。這是高分選手在考場上最得心應手的標準答題模板。

In [ ]:
# 範例 12.6.6：多重目標搜尋——收集全部匹配索引
scores = [85, 90, 72, 90, 60, 90, 88]
target = 90
print("全體學生成績清單：", scores)
print("搜尋目標分數：", target)

# 方法 1：使用傳統迴圈收集
all_indices = []
for idx, s in enumerate(scores):
    if s == target:
        all_indices.append(idx)

print("方法 1 收集到的索引清單：", all_indices)
print(f"分數為 {target} 的同學共有 {len(all_indices)} 位，座號分別是: {all_indices}")

# 方法 2：使用列表生成式一行完成
quick_indices = [i for i, s in enumerate(scores) if s == target]
print("方法 2 列表生成式極速收集：", quick_indices)

# 實戰輸出判斷：若找不到則輸出 -1
missing_target = 100
missing_indices = [i for i, s in enumerate(scores) if s == missing_target]
if missing_indices:
    print(*missing_indices)
else:
    print(f"搜尋 {missing_target} 分：找不到任何符合者，輸出 -1")

In [ ]:
# 填空題 12.6.6：收集所有不及格學生的座號
# 任務：找出所有分數小於 60 分的學生索引清單。
class_scores = [78, 55, 92, 48, 85, 59]

# 請使用列表生成式搭配 enumerate，條件為 s < 60
failed_indices = [___ for i, s in enumerate(class_scores) if ___ < 60]

print("不及格學生的座號索引清單：", failed_indices)
print(f"需要參加補考的同學人數：{len(failed_indices)} 人")

In [ ]:
# ==========================================
# [4] Code 練習題 12.6.6
# 任務說明：
# 給定一組整數數列 nums 與目標值 target。
# 請撰寫程式：
# 1. 收集所有數值等於 target 的索引位置放入 matches。
# 2. 若 matches 非空，印出 "匹配索引：X" 並印出總共出現次數。
# 3. 若 matches 為空，印出 "完全無匹配項目"。
#
# 【公開測試資料 1】
# nums = [12, 5, 8, 5, 20, 5]
# target = 5
# 預期輸出：
# 匹配索引： [1, 3, 5]
# 出現次數： 3
#
# 【公開測試資料 2】
# nums = [10, 20, 30]
# target = 99
# 預期輸出：
# 完全無匹配項目
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
nums = [12, 5, 8, 5, 20, 5]
target = 5

matches = [i for i, x in enumerate(nums) if x == target]
if matches:
    print("匹配索引：", matches)
    print("出現次數：", len(matches))
else:
    print("完全無匹配項目")

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.6.6
# 任務說明：
# 某氣象觀測站記錄了一週的降雨量（毫米）：rainfall = [0, 15, 0, 42, 0, 0, 8]
# 請寫出程式碼：
# 1. 找出所有「完全沒下雨（降雨量 == 0）」的星期索引（0 代表週一，依此類推）。
# 2. 找出所有「發生強降雨（降雨量 >= 30）」的星期索引。
# 3. 分別印出這兩組索引清單。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
rainfall = [0, 15, 0, 42, 0, 0, 8]
zero_rain_days = [i for i, r in enumerate(rainfall) if r == 0]
heavy_rain_days = [i for i, r in enumerate(rainfall) if r >= 30]

print("完全沒下雨的天數索引：", zero_rain_days)
print("發生強降雨的天數索引：", heavy_rain_days)

### 學習總結與通關回顧

恭喜你順利通關 **12.6 線性搜尋與 index() 例外防範**！

在本單元中，你奠定了搜尋演算法最扎實的工程基礎與避坑防線：
- **線性搜尋（Linear Search）核心**：
  - 面對無序資料，由頭至尾循序比對，是唯一全能的基礎搜尋演算法。
  - 時間複雜度：最佳 $O(1)$、最差 $O(N)$、平均約 $\frac{N+1}{2}$ 次比對。
- **`list.index(x)` 的致命雷區**：
  - 元素不存在時會無情拋出 `ValueError`，絕不會好心回傳 -1！
- **兩大安全查詢防禦策略**：
  - **哨兵模式**：`if x in a:` 先敲門確認，再呼叫 `a.index(x)`，最優雅直觀。
  - **例外捕捉**：`try ... except ValueError:` 攔截異常，杜絕 OJ 評判當機。
- **手刻通用線性搜尋標準規範**：
  - 找到時回傳命中索引，走完全部仍無結果時回傳 `-1`。
- **多重目標索引收集**：
  - `[i for i, val in enumerate(data) if val == target]` 一行收集全體匹配位置。

---
**下一關預告**：如果串列已經排好序，難道我們還要像盲人摸象一樣一間一間敲門搜尋嗎？能不能像猜數字遊戲一樣，每猜一次就把搜尋範圍直接砍掉一半？下一節 **12.7 二分搜尋法手刻演算法一：猜數字模型與精確匹配** 將帶你見證 $O(\log N)$ 的對數級神速飛躍！